<a href="https://colab.research.google.com/github/emiliebumatay/portfolio/blob/main/axiom_strat_spend_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Define the file path
file_path = 'axiom_strat_spend_raw_2023-2025.xlsx'

# Load each sheet into its own DataFrame
df_transactions = pd.read_excel(file_path, sheet_name='ERP_Raw_Dump')
df_budget = pd.read_excel(file_path, sheet_name='Dim_Budgets')

# Verify the data loaded correctly
print("✅ Success! Both sheets loaded independently.")
print(f"Transactions DataFrame: {df_transactions.shape[0]} rows, {df_transactions.shape[1]} columns")
print(f"Budget DataFrame: {df_budget.shape[0]} rows, {df_budget.shape[1]} columns")

✅ Success! Both sheets loaded independently.
Transactions DataFrame: 1285 rows, 11 columns
Budget DataFrame: 12 rows, 3 columns


In [2]:
# Strip out 'PHP' and commas from the RAW_AMOUNT column
df_transactions['RAW_AMOUNT'] = (
    df_transactions['RAW_AMOUNT']
    .astype(str)
    .str.replace('PHP', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

# Convert the cleaned text into a true numeric data type
# Any missing values or weird blanks will safely be turned into NaN (Not a Number)
df_transactions['RAW_AMOUNT'] = pd.to_numeric(df_transactions['RAW_AMOUNT'], errors='coerce')

# Verify our transformation worked perfectly
print("--- Check RAW_AMOUNT Data Type ---")
print(df_transactions['RAW_AMOUNT'].dtype)

print("\n--- First 5 Cleaned Rows ---")
print(df_transactions['RAW_AMOUNT'].head())

--- Check RAW_AMOUNT Data Type ---
float64

--- First 5 Cleaned Rows ---
0    282706.32
1    115160.76
2    104166.79
3     90582.74
4    111666.30
Name: RAW_AMOUNT, dtype: float64


In [3]:
# Define the columns we want to remove
columns_to_drop = ['TAX_RATE', 'INVOICE_REF']

# Drop them from our DataFrame programmatically
# errors='ignore' ensures that if you run this cell twice by mistake, it won't crash
df_transactions = df_transactions.drop(columns=columns_to_drop, errors='ignore')

# Verify our remaining column headers
print("--- Current Columns in Transactions ---")
print(df_transactions.columns.tolist())

--- Current Columns in Transactions ---
['RAW_PO_NUMBER', 'TRANSACTION_DATE', 'DEPARTMENT_NAME', 'EXPENSE_CATEGORY', 'VENDOR_NAME', 'RAW_AMOUNT', 'GL_CODE', 'APPROVAL_STATUS', 'SYSTEM_COMMENTS']


In [4]:
# Create a mapping dictionary to align messy names with official budget categories
department_mapping = {
    'it operations': 'IT Operations',
    'IT Operations': 'IT Operations',
    'MARKETING': 'Marketing & Sales',
    'Marketing & Sales': 'Marketing & Sales',
    'Facilities and Ops': 'Facilities & Ops',
    'Facilities & Ops': 'Facilities & Ops',
    'HR & Admin': 'HR & Admin'
}

# Apply the mapping to standardize the department column
df_transactions['DEPARTMENT_NAME'] = df_transactions['DEPARTMENT_NAME'].map(department_mapping)

# Check unique values to confirm everything is perfectly cleaned and uniform
print("--- Standardized Departments ---")
print(df_transactions['DEPARTMENT_NAME'].unique())

--- Standardized Departments ---
['IT Operations' 'Facilities & Ops' 'Marketing & Sales' 'HR & Admin' nan]


In [5]:
# Record the original row count before cleaning data
initial_rows = len(df_transactions)

# Drop rows ONLY if the RAW_AMOUNT itself is missing
df_transactions = df_transactions.dropna(subset=['RAW_AMOUNT'])

# Fill missing DEPARTMENT_NAME values with 'Unassigned'
df_transactions['DEPARTMENT_NAME'] = df_transactions['DEPARTMENT_NAME'].fillna('Unassigned')

# Calculate how many records were completely removed (the ones missing amounts)
rows_removed = initial_rows - len(df_transactions)

# Verify the current missing value counts
print(f"🧹 Removed {rows_removed} records missing a RAW_AMOUNT.")
print("✅ Remaining missing department names have been successfully flagged as 'Unassigned'.")
print("\n--- Remaining Missing Values Per Column ---")
print(df_transactions.isnull().sum())

🧹 Removed 20 records missing a RAW_AMOUNT.
✅ Remaining missing department names have been successfully flagged as 'Unassigned'.

--- Remaining Missing Values Per Column ---
RAW_PO_NUMBER       0
TRANSACTION_DATE    0
DEPARTMENT_NAME     0
EXPENSE_CATEGORY    0
VENDOR_NAME         0
RAW_AMOUNT          0
GL_CODE             0
APPROVAL_STATUS     0
SYSTEM_COMMENTS     0
dtype: int64


In [6]:
# Check for exact duplicate rows across the entire dataset
total_duplicates = df_transactions.duplicated().sum()
print(f"Total exact duplicate rows found: {total_duplicates}")

# If duplicates exist, let's look at a few sample rows to assess them
if total_duplicates > 0:
    print("\n--- Sample of Duplicate Rows ---")
    display(df_transactions[df_transactions.duplicated(keep=False)].head(6))

Total exact duplicate rows found: 34

--- Sample of Duplicate Rows ---


,RAW_PO_NUMBER,TRANSACTION_DATE,DEPARTMENT_NAME,EXPENSE_CATEGORY,VENDOR_NAME,RAW_AMOUNT,GL_CODE,APPROVAL_STATUS,SYSTEM_COMMENTS
48,,26-Mar-24,Marketing & Sales,Software Licenses,GlobalMedia Ads,80512.44,GL-500100,PENDING,System Auto-Generated
81,PO-2024-2082,24-Nov-25,Facilities & Ops,Hardware Procurement,PaperCorp Supplies,222770.35,GL-500100,PENDING,Urgent operational requirement!
101,PO-2024-2102,06/19/2023,IT Operations,Travel & Entertainment,Prime Space Realty,76337.26,GL-500200,PENDING,Urgent operational requirement!
173,PO-2024-2174,06/03/2025,Marketing & Sales,Digital Advertising,Prime Space Realty,109049.84,GL-600300,Approved,System Auto-Generated
250,PO-2024-2251,09-Jan-24,Facilities & Ops,Office Rent,Prime Space Realty,118863.16,GL-600300,REJECTED,System Auto-Generated
273,PO-2024-2274,25-Jan-23,Marketing & Sales,Office Rent,CloudSaaS Corp,78418.81,GL-500100,APPROVED,Urgent operational requirement!


In [7]:
# Let's see the exact breakdown of all text variations in that column
print("--- Raw Status Value Counts ---")
print(df_transactions['APPROVAL_STATUS'].value_counts(dropna=False))

--- Raw Status Value Counts ---
APPROVAL_STATUS
PENDING     330
REJECTED    317
APPROVED    314
Approved    304
Name: count, dtype: int64


In [8]:
# Record rows before filtering for approval status
rows_before_status_filter = len(df_transactions)

# Standardize the text by making it ALL CAPS and stripping extra spaces
df_transactions['APPROVAL_STATUS'] = df_transactions['APPROVAL_STATUS'].astype(str).str.strip().str.upper()

# CRITICAL FILTER: Keep only the standardized 'APPROVED' transactions
df_transactions = df_transactions[df_transactions['APPROVAL_STATUS'] == 'APPROVED']

# Calculate how many truly unapproved (pending/rejected) records were removed
unapproved_removed = rows_before_status_filter - len(df_transactions)

print(f"🛑 Successfully filtered out {unapproved_removed} 'PENDING' or 'REJECTED' records.")
print(f"✅ Kept {len(df_transactions)} total rows, and all are now uniformly labeled as 'APPROVED'.")

# Quick check to prove it worked perfectly
print("\n--- Current Status Values ---")
print(df_transactions['APPROVAL_STATUS'].value_counts())

🛑 Successfully filtered out 647 'PENDING' or 'REJECTED' records.
✅ Kept 618 total rows, and all are now uniformly labeled as 'APPROVED'.

--- Current Status Values ---
APPROVAL_STATUS
APPROVED    618
Name: count, dtype: int64


In [9]:
# Isolate all duplicates and sort them by date and vendor to place twins side-by-side
duplicate_df = df_transactions[df_transactions.duplicated(keep=False)]
sorted_duplicates = duplicate_df.sort_values(by=['TRANSACTION_DATE', 'VENDOR_NAME'])

print(f"Displaying paired duplicates ({len(sorted_duplicates)} rows total):")
display(sorted_duplicates.head(6))

Displaying paired duplicates (32 rows total):


,RAW_PO_NUMBER,TRANSACTION_DATE,DEPARTMENT_NAME,EXPENSE_CATEGORY,VENDOR_NAME,RAW_AMOUNT,GL_CODE,APPROVAL_STATUS,SYSTEM_COMMENTS
1031,PO-2024-3032,03/16/2025,IT Operations,Digital Advertising,GlobalMedia Ads,136809.69,GL-600300,APPROVED,System Auto-Generated
1263,PO-2024-3032,03/16/2025,IT Operations,Digital Advertising,GlobalMedia Ads,136809.69,GL-600300,APPROVED,System Auto-Generated
173,PO-2024-2174,06/03/2025,Marketing & Sales,Digital Advertising,Prime Space Realty,109049.84,GL-600300,APPROVED,System Auto-Generated
1252,PO-2024-2174,06/03/2025,Marketing & Sales,Digital Advertising,Prime Space Realty,109049.84,GL-600300,APPROVED,System Auto-Generated
1175,PO-2024-3176,10/09/2025,IT Operations,Travel & Entertainment,Prime Space Realty,65136.42,GL-500200,APPROVED,System Auto-Generated
1260,PO-2024-3176,10/09/2025,IT Operations,Travel & Entertainment,Prime Space Realty,65136.42,GL-500200,APPROVED,System Auto-Generated


In [10]:
# Drop exact duplicate rows, keeping only the first occurrence
df_transactions = df_transactions.drop_duplicates(keep='first')

# Reset the index so our row numbers remain sequential and clean
df_transactions = df_transactions.reset_index(drop=True)

print(f"✅ Success! 31 duplicate rows removed.")
print(f"Remaining active records in dataset: {len(df_transactions)} rows")

✅ Success! 31 duplicate rows removed.
Remaining active records in dataset: 602 rows


In [11]:
# Convert the transaction date column into a uniform datetime format
df_transactions['TRANSACTION_DATE'] = pd.to_datetime(df_transactions['TRANSACTION_DATE'], errors='coerce')

# Extract the year to create a clean link to the budget table
df_transactions['FISCAL_YEAR'] = df_transactions['TRANSACTION_DATE'].dt.year

# Let's verify the new column and check our date range
print("✅ Success! Date formats standardized.")
print("\n--- Value Counts for Our New Fiscal Year Column ---")
print(df_transactions['FISCAL_YEAR'].value_counts().sort_index())

print("\n--- Sample of Cleaned Dates and Years ---")
display(df_transactions[['TRANSACTION_DATE', 'FISCAL_YEAR']].head())

✅ Success! Date formats standardized.

--- Value Counts for Our New Fiscal Year Column ---
FISCAL_YEAR
2023    194
2024    223
2025    185
Name: count, dtype: int64

--- Sample of Cleaned Dates and Years ---


/tmp/ipykernel_20782/2332605405.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_transactions['TRANSACTION_DATE'] = pd.to_datetime(df_transactions['TRANSACTION_DATE'], errors='coerce')


,TRANSACTION_DATE,FISCAL_YEAR
0,2025-10-02,2025
1,2025-09-08,2025
2,2025-10-24,2025
3,2025-10-16,2025
4,2023-05-23,2023


In [12]:
# Aggregate actual transaction spend by Year and Department
df_actual_spend = (
    df_transactions.groupby(['FISCAL_YEAR', 'DEPARTMENT_NAME'])['RAW_AMOUNT']
    .sum()
    .reset_index()
)
df_actual_spend.columns = ['fiscal_year', 'department', 'actual_spend_php']

# Standardize column names on the budget sheet to lowercase to match
df_budget_clean = df_budget.copy()
df_budget_clean.columns = df_budget_clean.columns.str.lower()
df_budget_clean['department'] = df_budget_clean['department'].str.strip()

# Use an OUTER join to ensure 'Unassigned' rows are captured
df_variance_analysis = pd.merge(
    df_budget_clean,
    df_actual_spend,
    on=['fiscal_year', 'department'],
    how='outer'
)

# Handle missing values resulting from the outer join
# If a department/year has no budget (like Unassigned), set it to 0
df_variance_analysis['allocated_budget_php'] = df_variance_analysis['allocated_budget_php'].fillna(0)
df_variance_analysis['actual_spend_php'] = df_variance_analysis['actual_spend_php'].fillna(0)

# Fill missing years for Unassigned records if they appear blank after the outer join
df_variance_analysis['fiscal_year'] = df_variance_analysis['fiscal_year'].astype(int)

# Calculate the financial metrics: Variance Amount and Variance Percentage
df_variance_analysis['variance_php'] = (
    df_variance_analysis['actual_spend_php'] - df_variance_analysis['allocated_budget_php']
)
df_variance_analysis['variance_percentage'] = (
    (df_variance_analysis['variance_php'] / df_variance_analysis['allocated_budget_php']) * 100
)

# Replace infinite percentages (where budget was 0) with a clean 100.0 flag
df_variance_analysis['variance_percentage'] = (
    df_variance_analysis['variance_percentage']
    .replace([float('inf'), -float('inf')], 100.0)
)

# Display the final executive summary table
print("--- 📊 FINAL EXECUTIVE VARIANCE REPORT (INCLUDING UNASSIGNED LEAKAGE) ---")
display(df_variance_analysis.sort_values(by=['fiscal_year', 'department']).round(2))

# Verify the true aggregate corporate totals
total_overspend = df_variance_analysis['variance_php'].sum()
print("\n--- 🧾 Overall Corporate Sanity Check ---")
print(f"True Combined Variance: PHP {total_overspend:,.2f}")

--- 📊 FINAL EXECUTIVE VARIANCE REPORT (INCLUDING UNASSIGNED LEAKAGE) ---


,fiscal_year,department,allocated_budget_php,actual_spend_php,variance_php,variance_percentage
0,2023,Facilities & Ops,5800000.0,6190664.82,390664.82,6.74
1,2023,HR & Admin,4100000.0,2251691.55,-1848308.45,-45.08
2,2023,IT Operations,6500000.0,7988971.44,1488971.44,22.91
3,2023,Marketing & Sales,5200000.0,5594418.60,394418.60,7.58
4,2023,Unassigned,0.0,1113637.31,1113637.31,100.00
5,2024,Facilities & Ops,6140000.0,8618590.64,2478590.64,40.37
6,2024,HR & Admin,4340000.0,3493418.25,-846581.75,-19.51
7,2024,IT Operations,6890000.0,6640728.11,-249271.89,-3.62
8,2024,Marketing & Sales,5510000.0,9007624.48,3497624.48,63.48
9,2024,Unassigned,0.0,690655.09,690655.09,100.00



--- 🧾 Overall Corporate Sanity Check ---
True Combined Variance: PHP 9,344,600.07


In [13]:
# Check how many transactions are now sitting in our new 'Unassigned' category
print("--- Current Department Distribution ---")
print(df_transactions['DEPARTMENT_NAME'].value_counts())

--- Current Department Distribution ---
DEPARTMENT_NAME
Facilities & Ops     188
Marketing & Sales    158
IT Operations        152
HR & Admin            78
Unassigned            26
Name: count, dtype: int64


In [14]:
# Sort the variance analysis table by year and department for a clean presentation
df_variance_sorted = df_variance_analysis.sort_values(by=['fiscal_year', 'department'])

# Export both DataFrames into your custom-named master workbook
with pd.ExcelWriter('axiom_strat_spend_clean_2023-2025.xlsx', engine='openpyxl') as writer:
    # Tab 1: High-level Variance Analysis
    df_variance_sorted.to_excel(writer, sheet_name='Executive Summary', index=False)

    # Tab 2: Granular Cleaned Transactions
    df_transactions.to_excel(writer, sheet_name='Cleaned Transactions', index=False)

print("Filename: 'axiom_strat_spend_clean_2023-2025.xlsx' is ready in your files tab.")

Filename: 'axiom_strat_spend_clean_2023-2025.xlsx' is ready in your files tab.
